# rerank_llm_nqs — NQS-expanded judge input (the untried reranking surface)

**Motivation.** NQS (Neural Query Synthesis) raises recall for implicit-diagnosis topics by appending
Qwen-synthesized diagnoses to the *retrieval* query — e.g. for Topic 43 (rash + oral ulcers) the
expansion includes 'Behçet's disease', tripling recall (0.154→0.462). But the reranker currently
receives the *original* patient note, so the LLM judge still sees 'rash + oral ulcers' when scoring
Behçet's trials → NDCG@10 = 0.071 despite adequate recall (§8a §2h). `condition_match_exp` applied
the same expansion to SapBERT cosine but a scalar cosine can't bridge an eponymous terminology gap
the way the LLM judge can once it reads the diagnosis name.

**What this notebook does.** Loads the NQS expansions from `nqs_expansions.jsonl` (written by
`nqs_retrieval.ipynb`), constructs an *NQS-augmented topic*
`topic_text + '\n\nLikely diagnoses: ' + expansion`, and re-scores the NQS pool with the LLM judge
using that richer query. Everything else is identical to `rerank_llm_feature` — same judge, same
trial representation (elig_first-L512), same yes/no logprob margin.

**Output.** `llm_scores_nqs_expanded.jsonl` — use as `llm_yesno_nqs` in `train_ensemble_full`
alongside the existing `llm_yesno` (different-input, not different-model: genuinely new signal).

**Anti-gaming.** Develop and gate on **TREC21 CV only**. Watch the implicit-diagnosis topic class
(TREC21 analogs: T55 Wilson's, T40/T74 Paget's, T52 Cholera, T32 HUS). Touch TREC22 once, only
after a meaningful TREC21 CV gain (≥+0.005). Do NOT report per-topic TREC22 numbers during development.

## Setup (Colab — GPU for Qwen-7B, ~14 GB fp16)

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q transformers accelerate datasets pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval, llm_yesno_scores, ndcg_at_k

# MUST be 'nqs' — we score the NQS pool with the NQS-expanded query.
POOL_TAG = 'nqs'
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag=POOL_TAG)
print('repr:', cfg.repr_tag(), '| judge:', cfg.llm_ckpt, '| pool:', POOL_TAG)

## Load pool + expansions

In [ ]:
SETS = ['trec21', 'kz', 'trec22']
OUT_PATH = cfg.path('data/llm_scores_nqs_expanded.jsonl')

corpus_ids, corpus_fields = load_corpus(cfg)
id2fields = dict(zip(corpus_ids, corpus_fields))
sets = load_eval(cfg, SETS)

# NQS pool (written by nqs_retrieval.ipynb).
nqs_pool = json.load(open(cfg.pool_path()))  # pool_tag='nqs'
pools = {
    s: {t: [d for d in docs[:cfg.llm_top_k] if d in id2fields]
        for t, docs in nqs_pool[s].items()}
    for s in SETS
}
print('pool sizes:', {s: sum(len(v) for v in p.values()) for s, p in pools.items()})

# NQS expansions (written by nqs_retrieval.ipynb cell-6).
EXP_PATH = cfg.path('data/nqs_expansions.jsonl')
if not os.path.exists(EXP_PATH):
    raise FileNotFoundError(
        f'Missing {EXP_PATH} — run nqs_retrieval.ipynb first (the expansion cell).'
    )
expansions = {}
for line in open(EXP_PATH):
    r = json.loads(line)
    expansions[(r['source'], r['topic_id'])] = r['expansion']
print(f'Loaded {len(expansions)} NQS expansions.')

# Spot-check: implicit-diagnosis TREC21 development targets.
TREC21_IMPLICIT = {'55': 'Wilson\'s disease', '40': 'Paget\'s', '74': 'Paget\'s',
                   '52': 'Cholera', '32': 'HUS/TTP'}
print('\nExpansion spot-check (TREC21 implicit-diagnosis targets):')
for tid, dx in TREC21_IMPLICIT.items():
    key = ('trec21', tid)
    if key in expansions:
        print(f'  T{tid} ({dx}): {expansions[key][:120]}')
    else:
        print(f'  T{tid}: NOT IN EXPANSIONS (run nqs_retrieval with SETS including trec21)')

## Load judge

In [ ]:
tok = AutoTokenizer.from_pretrained(cfg.llm_ckpt, padding_side='left')
if tok.pad_token is None: tok.pad_token = tok.eos_token
llm = AutoModelForCausalLM.from_pretrained(
    cfg.llm_ckpt, torch_dtype=torch.float16, device_map='auto'
).eval()
print('judge loaded')

## Score with NQS-expanded query

The only change from `rerank_llm_feature`: the topic passed to `llm_yesno_scores` is
`original_note + '\n\nLikely diagnoses: ' + nqs_expansion` instead of the raw note.
The judge then reads 'rash + oral ulcers … Likely diagnoses: Behçet's disease, …'
and can correctly assess Behçet's trials.

In [ ]:
from tqdm.auto import tqdm

def nqs_topic(source, topic_id, raw_text):
    exp = expansions.get((source, topic_id), '')
    if exp:
        return raw_text + '\n\nLikely diagnoses: ' + exp
    return raw_text  # fallback: no expansion available

with open(OUT_PATH, 'w') as out:
    for s in SETS:
        for t, docs in tqdm(pools[s].items(), desc=f'judge {s}'):
            raw = sets[s]['topic2text'][t]
            augmented = nqs_topic(s, t, raw)
            scores = llm_yesno_scores(
                llm, tok, augmented,
                [id2fields[d] for d in docs], cfg, batch=8
            )
            for d, sc in zip(docs, scores):
                out.write(json.dumps({
                    'source': s, 'topic_id': t, 'doc_id': d,
                    'llm_score': float(sc)
                }) + '\n')
print('wrote', OUT_PATH)

## Standalone NDCG + implicit-diagnosis per-topic check

In [ ]:
# Load scores.
nqs_exp_scores = {}
for line in open(OUT_PATH):
    r = json.loads(line)
    nqs_exp_scores.setdefault((r['source'], r['topic_id']), {})[r['doc_id']] = r['llm_score']

# Also load baseline llm_scores (original judge on R pool) for comparison.
BASE_PATH = cfg.path('data/llm_scores_nqs.jsonl')  # judge on NQS pool, original topic text
base_scores = {}
if os.path.exists(BASE_PATH):
    for line in open(BASE_PATH):
        r = json.loads(line)
        base_scores.setdefault((r['source'], r['topic_id']), {})[r['doc_id']] = r['llm_score']
    print('baseline (NQS pool, original topic) loaded')
else:
    print('No NQS-pool baseline found — will show NQS-expanded standalone only.')

# Per-split NDCG.
rows = []
for s in SETS:
    rel = sets[s]['rel_dict']; vals_exp, vals_base = [], []
    for t, docs in pools[s].items():
        if (s, t) not in nqs_exp_scores: continue
        ranked_exp = sorted(docs, key=lambda d: nqs_exp_scores[(s, t)].get(d, -999), reverse=True)
        vals_exp.append(ndcg_at_k(ranked_exp, rel[t]))
        if base_scores:
            ranked_base = sorted(docs, key=lambda d: base_scores.get((s, t), {}).get(d, -999), reverse=True)
            vals_base.append(ndcg_at_k(ranked_base, rel[t]))
    row = {'split': s, 'nqs_exp_ndcg@10': round(float(np.mean(vals_exp)), 4)}
    if vals_base: row['base_ndcg@10'] = round(float(np.mean(vals_base)), 4)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Per-topic check on TREC21 implicit-diagnosis development targets.
# These are the TREC21 analogs of Topic 43's eponymous-terminology failure mode.
# We do NOT look at TREC22 per-topic here (anti-gaming).
print('TREC21 implicit-diagnosis topics (development check, NOT the TREC22 test):')
print()
s = 'trec21'
rel = sets[s]['rel_dict']
rows = []
for tid, dx_note in TREC21_IMPLICIT.items():
    docs = pools[s].get(tid, [])
    if not docs or (s, tid) not in nqs_exp_scores: continue
    ranked_exp = sorted(docs, key=lambda d: nqs_exp_scores[(s, tid)].get(d, -999), reverse=True)
    ndcg_exp = ndcg_at_k(ranked_exp, rel.get(tid, {}))
    row = {'topic': tid, 'dx_note': dx_note, 'nqs_exp_ndcg@10': round(ndcg_exp, 3)}
    if base_scores and (s, tid) in base_scores:
        ranked_base = sorted(docs, key=lambda d: base_scores[(s, tid)].get(d, -999), reverse=True)
        row['base_ndcg@10'] = round(ndcg_at_k(ranked_base, rel.get(tid, {})), 3)
        row['delta'] = round(ndcg_exp - row['base_ndcg@10'], 3)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
print()
print('(Gains here = mechanism confirmed on TREC21 implicit-diagnosis class.)')

## Use in train_ensemble_full

Add `llm_scores_nqs_expanded.jsonl` as an additional feature `llm_yesno_nqs` alongside the existing
`llm_yesno` in `train_ensemble_full` (with `POOL_TAG='nqs'`). The feature key in the feature-loading
cell:

```python
NQS_EXP_PATH = cfg.path('data/llm_scores_nqs_expanded.jsonl')
llm_nqs = {}
if os.path.exists(NQS_EXP_PATH):
    for l in open(NQS_EXP_PATH):
        r = json.loads(l)
        llm_nqs[(r['source'], r['topic_id'], r['doc_id'])] = r['llm_score']
    FEATURES += ['llm_yesno_nqs']
    print('llm_yesno_nqs loaded:', len(llm_nqs))
```

And in `featvec`:
```python
if llm_nqs: v['llm_yesno_nqs'] = llm_nqs.get((s, t, d), cfg.llm_floor)
```

**Gate (anti-gaming):** TREC21 CV must improve by ≥+0.005 before touching TREC22.

**What to report:**
- If gain on TREC21 CV: note that the gain concentrates on implicit-diagnosis topics vs. the mean
- If null: the expansion doesn't add over the existing `llm_yesno`
  → check correlation `r(llm_yesno, llm_yesno_nqs)` on the TREC21 pool: if r > 0.90, genuinely redundant
    (judge already handles the gap zero-shot on the NQS pool); if r < 0.70, something else is wrong